# 04 — Modeling and Tuning

This notebook compares Ridge Regression and Histogram Gradient Boosting with grouped cross-validation, evaluates feature selection versus SVD, performs hyperparameter tuning, and saves the tuned estimators for final evaluation.

### Imports — why this cell is needed
Loads the modelling, validation, tuning, and serialization libraries.

In [1]:
import re
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectPercentile, f_regression
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import (
    GroupShuffleSplit,
    GroupKFold,
    cross_validate,
    GridSearchCV,
    RandomizedSearchCV
)
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_squared_log_error,
    r2_score
)

import joblib

### Repository paths — why this cell is needed
Loads engineered data and saves fitted models using relative paths.

In [2]:
def find_repo_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Run this notebook from inside the cloned "
        "CSE437 repository, with data/ and notebooks/ folders present."
    )

ROOT = find_repo_root()
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
FIGURES_DIR = ROOT / "figures"
MODELS_DIR = ROOT / "models"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root:", ROOT)

Repository root: /content/cse437-building-energy-G9


### Load engineered data — why this cell is needed
Uses the output of `03_feature_engineering.ipynb`.

In [3]:
ENGINEERED_CSV = PROCESSED_DIR / "seattle_energy_features.csv"
if not ENGINEERED_CSV.exists():
    raise FileNotFoundError(
        "Run 03_feature_engineering.ipynb first. "
        "Expected data/processed/seattle_energy_features.csv"
    )

df = pd.read_csv(ENGINEERED_CSV, low_memory=False)
print("Loaded:", ENGINEERED_CSV.relative_to(ROOT))
print("Shape:", df.shape)

Loaded: data/processed/seattle_energy_features.csv
Shape: (34822, 29)


### Grouped development/test split — why this cell is needed
Keeps all reporting years of the same building together.

In [4]:
TARGET = "SiteEUI"
GROUP = "OSEBuildingID"

X = df.drop(columns=[TARGET, GROUP])
y = df[TARGET]
groups = df[GROUP]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

dev_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_dev = X.iloc[dev_idx].reset_index(drop=True)
y_dev = y.iloc[dev_idx].reset_index(drop=True)
groups_dev = groups.iloc[dev_idx].reset_index(drop=True)

X_test = X.iloc[test_idx].reset_index(drop=True)
y_test = y.iloc[test_idx].reset_index(drop=True)
groups_test = groups.iloc[test_idx].reset_index(drop=True)

shared_buildings = set(groups_dev).intersection(set(groups_test))

print("Development rows:", len(X_dev))
print("Test rows:", len(X_test))
print("Development buildings:", groups_dev.nunique())
print("Test buildings:", groups_test.nunique())
print("Shared building IDs:", len(shared_buildings))

assert len(shared_buildings) == 0

Development rows: 27807
Test rows: 7015
Development buildings: 3020
Test buildings: 756
Shared building IDs: 0


### Define preprocessing — why this cell is needed
Places clipping, imputation, scaling, and encoding inside the modelling pipeline.

In [5]:
numeric_features = X_dev.select_dtypes(include=np.number).columns.tolist()
categorical_features = [
    c for c in X_dev.columns
    if c not in numeric_features
]


for col in categorical_features:
    X_dev[col] = X_dev[col].astype(object)
    X_test[col] = X_test[col].astype(object)
    X_dev[col] = X_dev[col].where(pd.notna(X_dev[col]), np.nan)
    X_test[col] = X_test[col].where(pd.notna(X_test[col]), np.nan)

class QuantileClipper(BaseEstimator, TransformerMixin):
    def __init__(self, columns=None, lower=0.01, upper=0.99):
        self.columns = columns
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = X.copy()
        self.bounds_ = {}
        for col in self.columns or []:
            series = pd.to_numeric(X[col], errors="coerce")
            self.bounds_[col] = (
                series.quantile(self.lower),
                series.quantile(self.upper)
            )
        return self

    def transform(self, X):
        X = X.copy()
        for col, (low, high) in self.bounds_.items():
            if col in X.columns:
                X[col] = pd.to_numeric(
                    X[col], errors="coerce"
                ).clip(low, high)
        return X

def make_preprocessor():
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(
            strategy="median",
            keep_empty_features=True
        )),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(
            strategy="most_frequent",
            keep_empty_features=True
        )),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=10,
            sparse_output=False
        ))
    ])

    return ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ], remainder="drop")

print("Numeric predictors:", len(numeric_features))
print("Categorical predictors:", len(categorical_features))
print("Missing development cells before preprocessing:",
      int(X_dev.isna().sum().sum()))

Numeric predictors: 21
Categorical predictors: 6
Missing development cells before preprocessing: 103655


### Prepare a development matrix for SVD sizing — why this cell is needed
Determines a safe number of SVD components without using the test set.

In [6]:
clip_demo = QuantileClipper(
    columns=numeric_features,
    lower=0.01,
    upper=0.99
).fit(X_dev)

X_clip_demo = clip_demo.transform(X_dev)
preprocessor_demo = make_preprocessor()
X_prepared_demo = preprocessor_demo.fit_transform(X_clip_demo, y_dev)

max_components = max(2, min(30, X_prepared_demo.shape[1] - 1))
print("Maximum SVD components considered:", max_components)

Maximum SVD components considered: 30


### Define the two model families — why this cell is needed
Defines Ridge Regression and Histogram Gradient Boosting, plus the SVD/feature-selection Ridge representations.

In [7]:
def make_ridge_selected(alpha=1.0, percentile=70):
    return Pipeline([
        ("clip", QuantileClipper(
            numeric_features, 0.01, 0.99
        )),
        ("preprocess", make_preprocessor()),
        ("select", SelectPercentile(
            f_regression,
            percentile=percentile
        )),
        ("model", TransformedTargetRegressor(
            regressor=Ridge(alpha=alpha),
            func=np.log1p,
            inverse_func=np.expm1,
            check_inverse=False
        ))
    ])

def make_ridge_svd(
    alpha=1.0,
    n_components=20
):
    safe_components = min(
        n_components,
        max(2, X_prepared_demo.shape[1] - 1)
    )

    return Pipeline([
        ("clip", QuantileClipper(
            numeric_features, 0.01, 0.99
        )),
        ("preprocess", make_preprocessor()),
        ("svd", TruncatedSVD(
            n_components=safe_components,
            random_state=RANDOM_STATE
        )),
        ("model", TransformedTargetRegressor(
            regressor=Ridge(alpha=alpha),
            func=np.log1p,
            inverse_func=np.expm1,
            check_inverse=False
        ))
    ])

def make_boosting(
    percentile=70,
    learning_rate=0.05,
    max_iter=300,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    l2_regularization=1.0
):
    return Pipeline([
        ("clip", QuantileClipper(
            numeric_features, 0.01, 0.99
        )),
        ("preprocess", make_preprocessor()),
        ("select", SelectPercentile(
            f_regression,
            percentile=percentile
        )),
        ("model", TransformedTargetRegressor(
            regressor=HistGradientBoostingRegressor(
                loss="absolute_error",
                learning_rate=learning_rate,
                max_iter=max_iter,
                max_leaf_nodes=max_leaf_nodes,
                min_samples_leaf=min_samples_leaf,
                l2_regularization=l2_regularization,
                random_state=RANDOM_STATE
            ),
            func=np.log1p,
            inverse_func=np.expm1,
            check_inverse=False
        ))
    ])

### Initial grouped cross-validation comparison — why this cell is needed
Compares train and validation MAE, RMSE, and R² without contaminating buildings across folds.

In [8]:
cv = GroupKFold(n_splits=3)

models = {
    "Ridge + Feature Selection":
        make_ridge_selected(),
    "Ridge + SVD":
        make_ridge_svd(
            n_components=min(20, max_components)
        ),
    "Gradient Boosting + Feature Selection":
        make_boosting()
}

comparison_rows = []

for name, model in models.items():
    print("Evaluating:", name)

    scores = cross_validate(
        model,
        X_dev,
        y_dev,
        groups=groups_dev,
        cv=cv,
        scoring={
            "MAE": "neg_mean_absolute_error",
            "RMSE": "neg_root_mean_squared_error",
            "R2": "r2"
        },
        return_train_score=True,
        n_jobs=1
    )

    comparison_rows.append({
        "Model": name,
        "Train MAE":
            -scores["train_MAE"].mean(),
        "Validation MAE":
            -scores["test_MAE"].mean(),
        "Train RMSE":
            -scores["train_RMSE"].mean(),
        "Validation RMSE":
            -scores["test_RMSE"].mean(),
        "Train R2":
            scores["train_R2"].mean(),
        "Validation R2":
            scores["test_R2"].mean(),
        "MAE Gap":
            (
                -scores["test_MAE"].mean()
                + scores["train_MAE"].mean()
            )
    })

model_comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values("Validation MAE")
    .reset_index(drop=True)
)

display(model_comparison)

Evaluating: Ridge + Feature Selection


Evaluating: Ridge + SVD


Evaluating: Gradient Boosting + Feature Selection


,Model,Train MAE,Validation MAE,Train RMSE,Validation RMSE,Train R2,Validation R2,MAE Gap
0,Gradient Boosting + Feature Selection,11.153375,18.352252,27.496213,37.052693,0.678223,0.413340,7.198877
1,Ridge + Feature Selection,17.677382,19.407159,33.943271,37.581349,0.508612,0.394940,1.729777
2,Ridge + SVD,22.433359,22.620400,46.476532,46.534139,0.079379,0.069414,0.187041


### Diagnose initial overfitting/underfitting — why this cell is needed
Uses train-versus-validation gaps to characterize generalization.

In [9]:
diagnosis_rows = []

for _, row in model_comparison.iterrows():
    r2_gap = row["Train R2"] - row["Validation R2"]

    if (
        row["Train MAE"] < 0.75 * row["Validation MAE"]
        and r2_gap > 0.15
    ):
        diagnosis = "Evidence of overfitting"
    elif (
        row["Train R2"] < 0.30
        and row["Validation R2"] < 0.30
    ):
        diagnosis = (
            "Underfitting / limited predictive signal"
        )
    else:
        diagnosis = (
            "No strong overfitting signal"
        )

    diagnosis_rows.append({
        "Model": row["Model"],
        "Train R2": row["Train R2"],
        "Validation R2": row["Validation R2"],
        "R2 Gap": r2_gap,
        "Diagnosis": diagnosis
    })

diagnosis_table = pd.DataFrame(diagnosis_rows)
display(diagnosis_table)

,Model,Train R2,Validation R2,R2 Gap,Diagnosis
0,Gradient Boosting + Feature Selection,0.678223,0.413340,0.264883,Evidence of overfitting
1,Ridge + Feature Selection,0.508612,0.394940,0.113672,No strong overfitting signal
2,Ridge + SVD,0.079379,0.069414,0.009964,Underfitting / limited predictive signal


### Choose feature selection versus SVD for Ridge — why this cell is needed
Selects the representation using grouped-validation MAE.

In [10]:
ridge_fs_mae = model_comparison.loc[
    model_comparison["Model"]
        == "Ridge + Feature Selection",
    "Validation MAE"
].iloc[0]

ridge_svd_mae = model_comparison.loc[
    model_comparison["Model"]
        == "Ridge + SVD",
    "Validation MAE"
].iloc[0]

if ridge_fs_mae <= ridge_svd_mae:
    ridge_representation = "feature_selection"
    print(
        "Feature selection is retained for Ridge "
        "because it produced lower grouped-validation MAE."
    )
else:
    ridge_representation = "svd"
    print(
        "SVD is retained for Ridge because it "
        "produced lower grouped-validation MAE."
    )

print("Ridge + Feature Selection validation MAE:",
      round(ridge_fs_mae, 3))
print("Ridge + SVD validation MAE:",
      round(ridge_svd_mae, 3))

Feature selection is retained for Ridge because it produced lower grouped-validation MAE.
Ridge + Feature Selection validation MAE: 19.407
Ridge + SVD validation MAE: 22.62


## Hyperparameter tuning

### Tune Ridge with GridSearchCV — why this cell is needed
Searches the exact Ridge hyperparameter space reported in the project.

In [11]:
if ridge_representation == "feature_selection":
    ridge_tuning_model = make_ridge_selected()

    ridge_search_space = {
        "select__percentile":
            [40, 50, 60, 70, 80, 90, 100],
        "model__regressor__alpha":
            [0.01, 0.1, 1.0, 10.0, 100.0]
    }
else:
    ridge_tuning_model = make_ridge_svd()

    candidate_components = sorted(set([
        max(2, min(10, max_components)),
        max(2, min(20, max_components)),
        max(2, min(30, max_components))
    ]))

    ridge_search_space = {
        "svd__n_components":
            candidate_components,
        "model__regressor__alpha":
            [0.01, 0.1, 1.0, 10.0, 100.0]
    }

print("Ridge Grid Search space:")
print(ridge_search_space)

ridge_grid = GridSearchCV(
    estimator=ridge_tuning_model,
    param_grid=ridge_search_space,
    scoring="neg_mean_absolute_error",
    cv=GroupKFold(n_splits=3),
    n_jobs=-1,
    refit=True,
    return_train_score=True
)

ridge_grid.fit(
    X_dev,
    y_dev,
    groups=groups_dev
)

print("\nBest Ridge parameters:")
print(ridge_grid.best_params_)
print(
    "Best Ridge validation MAE:",
    round(-ridge_grid.best_score_, 3)
)

ridge_tuning_results = (
    pd.DataFrame(ridge_grid.cv_results_)
)
ridge_tuning_results["Validation MAE"] = (
    -ridge_tuning_results["mean_test_score"]
)
ridge_tuning_results["Train MAE"] = (
    -ridge_tuning_results["mean_train_score"]
)

display(
    ridge_tuning_results
    .sort_values("rank_test_score")
    [[
        "rank_test_score",
        "params",
        "Train MAE",
        "Validation MAE",
        "std_test_score"
    ]]
    .head(10)
)

Ridge Grid Search space:
{'select__percentile': [40, 50, 60, 70, 80, 90, 100], 'model__regressor__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]}



Best Ridge parameters:
{'model__regressor__alpha': 10.0, 'select__percentile': 50}
Best Ridge validation MAE: 19.049


,rank_test_score,params,Train MAE,Validation MAE,std_test_score
22,1,"{'model__regressor__alpha': 10.0, 'select__per...",18.010963,19.049001,1.349595
15,2,"{'model__regressor__alpha': 1.0, 'select__perc...",17.908879,19.154593,1.322331
23,3,"{'model__regressor__alpha': 10.0, 'select__per...",17.890012,19.168120,1.314304
8,4,"{'model__regressor__alpha': 0.1, 'select__perc...",17.894250,19.207821,1.285543
21,5,"{'model__regressor__alpha': 10.0, 'select__per...",18.204511,19.209187,1.430840
24,6,"{'model__regressor__alpha': 10.0, 'select__per...",17.823128,19.216164,1.365321
1,7,"{'model__regressor__alpha': 0.01, 'select__per...",17.891115,19.221019,1.271735
25,8,"{'model__regressor__alpha': 10.0, 'select__per...",17.765951,19.262548,1.321343
14,9,"{'model__regressor__alpha': 1.0, 'select__perc...",18.087295,19.290662,1.413344
26,10,"{'model__regressor__alpha': 10.0, 'select__per...",17.720395,19.295240,1.288323


### Tune Gradient Boosting with RandomizedSearchCV — why this cell is needed
Searches the exact nonlinear-model hyperparameter space reported in the project.

In [12]:
boost_search_space = {
    "select__percentile":
        [40, 50, 60, 70, 80, 90, 100],
    "model__regressor__learning_rate":
        [0.02, 0.03, 0.05, 0.08, 0.12],
    "model__regressor__max_iter":
        [200, 300, 400, 500],
    "model__regressor__max_leaf_nodes":
        [15, 31, 63],
    "model__regressor__min_samples_leaf":
        [10, 20, 30, 40],
    "model__regressor__l2_regularization":
        [0.0, 0.5, 1.0, 2.0, 5.0]
}

print("Gradient Boosting Random Search space:")
print(boost_search_space)

boost_search = RandomizedSearchCV(
    estimator=make_boosting(),
    param_distributions=boost_search_space,
    n_iter=15,
    scoring="neg_mean_absolute_error",
    cv=GroupKFold(n_splits=3),
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
    verbose=1
)

boost_search.fit(
    X_dev,
    y_dev,
    groups=groups_dev
)

print("\nBest Gradient Boosting parameters:")
print(boost_search.best_params_)
print(
    "Best Gradient Boosting validation MAE:",
    round(-boost_search.best_score_, 3)
)

boost_tuning_results = (
    pd.DataFrame(boost_search.cv_results_)
)
boost_tuning_results["Validation MAE"] = (
    -boost_tuning_results["mean_test_score"]
)
boost_tuning_results["Train MAE"] = (
    -boost_tuning_results["mean_train_score"]
)

display(
    boost_tuning_results
    .sort_values("rank_test_score")
    [[
        "rank_test_score",
        "params",
        "Train MAE",
        "Validation MAE",
        "std_test_score"
    ]]
    .head(10)
)

Gradient Boosting Random Search space:
{'select__percentile': [40, 50, 60, 70, 80, 90, 100], 'model__regressor__learning_rate': [0.02, 0.03, 0.05, 0.08, 0.12], 'model__regressor__max_iter': [200, 300, 400, 500], 'model__regressor__max_leaf_nodes': [15, 31, 63], 'model__regressor__min_samples_leaf': [10, 20, 30, 40], 'model__regressor__l2_regularization': [0.0, 0.5, 1.0, 2.0, 5.0]}
Fitting 3 folds for each of 15 candidates, totalling 45 fits



Best Gradient Boosting parameters:
{'select__percentile': 90, 'model__regressor__min_samples_leaf': 10, 'model__regressor__max_leaf_nodes': 63, 'model__regressor__max_iter': 400, 'model__regressor__learning_rate': 0.02, 'model__regressor__l2_regularization': 5.0}
Best Gradient Boosting validation MAE: 18.283


,rank_test_score,params,Train MAE,Validation MAE,std_test_score
12,1,"{'select__percentile': 90, 'model__regressor__...",10.513119,18.282968,1.233754
8,2,"{'select__percentile': 100, 'model__regressor_...",11.650701,18.323090,1.275250
3,3,"{'select__percentile': 80, 'model__regressor__...",11.829662,18.348036,1.184622
6,4,"{'select__percentile': 80, 'model__regressor__...",12.997366,18.348817,1.306728
0,5,"{'select__percentile': 80, 'model__regressor__...",12.104658,18.370290,1.312514
13,6,"{'select__percentile': 80, 'model__regressor__...",10.453776,18.392033,1.315281
1,7,"{'select__percentile': 100, 'model__regressor_...",13.086656,18.414307,1.334156
11,8,"{'select__percentile': 100, 'model__regressor_...",13.701602,18.457745,1.390712
14,9,"{'select__percentile': 90, 'model__regressor__...",15.018480,18.512448,1.399691
7,10,"{'select__percentile': 60, 'model__regressor__...",8.269771,18.512668,1.313680


### Re-check tuned models — why this cell is needed
Reports train and grouped-validation performance after tuning.

In [13]:
tuned_models = {
    "Tuned Ridge":
        ridge_grid.best_estimator_,
    "Tuned Gradient Boosting":
        boost_search.best_estimator_
}

tuned_rows = []

for name, model in tuned_models.items():
    scores = cross_validate(
        model,
        X_dev,
        y_dev,
        groups=groups_dev,
        cv=GroupKFold(n_splits=3),
        scoring={
            "MAE": "neg_mean_absolute_error",
            "RMSE": "neg_root_mean_squared_error",
            "R2": "r2"
        },
        return_train_score=True,
        n_jobs=1
    )

    tuned_rows.append({
        "Model": name,
        "Train MAE":
            -scores["train_MAE"].mean(),
        "Validation MAE":
            -scores["test_MAE"].mean(),
        "Train RMSE":
            -scores["train_RMSE"].mean(),
        "Validation RMSE":
            -scores["test_RMSE"].mean(),
        "Train R2":
            scores["train_R2"].mean(),
        "Validation R2":
            scores["test_R2"].mean()
    })

tuned_comparison = (
    pd.DataFrame(tuned_rows)
    .sort_values("Validation MAE")
    .reset_index(drop=True)
)

display(tuned_comparison)

,Model,Train MAE,Validation MAE,Train RMSE,Validation RMSE,Train R2,Validation R2
0,Tuned Gradient Boosting,10.513119,18.282968,27.211903,37.203137,0.684709,0.409014
1,Tuned Ridge,18.010963,19.049001,34.985568,37.286768,0.478359,0.405058


### Save the tuned model bundle — why this cell is needed
Allows the evaluation notebook to run on a fresh kernel without repeating hyperparameter searches.

In [14]:
MODEL_BUNDLE = MODELS_DIR / "model_bundle.joblib"

bundle = {
    "ridge_model": ridge_grid.best_estimator_,
    "boost_model": boost_search.best_estimator_,
    "ridge_cv_mae": float(-ridge_grid.best_score_),
    "boost_cv_mae": float(-boost_search.best_score_),
    "ridge_best_params": ridge_grid.best_params_,
    "boost_best_params": boost_search.best_params_,
    "tuned_comparison": tuned_comparison,
    "random_state": RANDOM_STATE
}

joblib.dump(bundle, MODEL_BUNDLE)
print("Saved:", MODEL_BUNDLE.relative_to(ROOT))

Saved: models/model_bundle.joblib
